# Monitoring

In [ ]:
import json

import matplotlib.pyplot as plt
import mlflow
import nannyml as nml
import pandas as pd
import yaml
from IPython.display import Markdown, display

from churn_mlops.config import ServingSettings
from churn_mlops.data import load_raw_data, validate_data

# from churn_mlops.serving import load_model
from churn_mlops.tracking import ModelRegistry

pd.set_option("display.width", 150)

settings = ServingSettings()

## Reference and analysis data

* reference = training
* analysis = inference

In [ ]:
reference = load_raw_data(
    "predictions_training.csv", data_dir=settings.output_dir, drop_columns="customerid"
)
analysis = load_raw_data(
    "predictions_inference.csv", data_dir=settings.output_dir, drop_columns="customerid"
)

reference = validate_data(reference)
analysis = validate_data(analysis)

print(reference.shape)
print(analysis.shape)

In [ ]:
from churn_mlops.monitoring import build_monitoring_report

monitoring_report = build_monitoring_report(
    reference=reference,
    analysis=analysis,
    source_reference="batch",
    source_analysis="batch",
    drift=True,
    output_dir=settings.output_dir / "monitoring",
)
monitoring_report.summary

In [ ]:
print(reference.describe())

In [ ]:
print(analysis.describe())

In [ ]:
print(reference.describe(include="str"))

In [ ]:
print(analysis.describe(include="str"))

## Champion Model from MLflow Registry

In [ ]:
# loaded_model = load_model(
#     tracking_uri=settings.mlflow_tracking_uri,
#     model_name=settings.model_name,
#     model_alias=settings.model_alias,
# )
# loaded_model.model

In [ ]:
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
registry = ModelRegistry()
champion_model = registry.get_champion_version(model_name=settings.model_name)
champion_run = mlflow.get_run(champion_model.run_id)
champion_artifact_uri = champion_run.info.artifact_uri
champion_run_data = champion_run.data
champion_metrics = champion_run_data.metrics
champion_params = champion_run_data.params
champion_tags = champion_run_data.tags

In [ ]:
print(mlflow.artifacts.list_artifacts(artifact_uri=champion_artifact_uri))
champion_path_feature_artifact = mlflow.artifacts.list_artifacts(
    artifact_uri=champion_artifact_uri + "/features"
)[0].path
champion_path_config_artifact = mlflow.artifacts.list_artifacts(
    artifact_uri=champion_artifact_uri + "/config"
)[0].path
champion_path_docu_artifact = mlflow.artifacts.list_artifacts(
    artifact_uri=champion_artifact_uri + "/documentation"
)[0].path

In [ ]:
path_feature_names_json = champion_artifact_uri + "/" + champion_path_feature_artifact
path_config_artifact = champion_artifact_uri + "/" + champion_path_config_artifact
path_docu_artifact = champion_artifact_uri + "/" + champion_path_docu_artifact

In [ ]:
with open(path_docu_artifact, "r", encoding="utf-8") as f:
    content = f.read()
display(Markdown(content))

In [ ]:
with open(path_config_artifact, "r") as file:
    config = yaml.safe_load(file)
config

In [ ]:
with open(path_feature_names_json, "r") as fp:
    champion_features = json.load(fp)

target = champion_params["target_column"]

problem_type = "classification_binary"
col_pred_proba = "predicted_probability"
col_pred = "predicted_class"

timestamp = "reference_date"

features = champion_features["input"]

threshold = champion_params["threshold"]

len(champion_features["output"]) == int(champion_params["feature_count"])

## Univariate Drift Detection

In [ ]:
uvar_drift_calc = nml.UnivariateDriftCalculator(
    column_names=features,
    timestamp_column_name=timestamp,
    chunk_period="M",
    continuous_methods=["jensen_shannon"],
    categorical_methods=["jensen_shannon"],
)

uvar_drift = uvar_drift_calc.fit(reference).calculate(analysis)

uvar_drift.plot().show()

In [ ]:
# rank features by drift
uvar_drift_js = uvar_drift.filter(metrics="jensen_shannon").to_df()
uvar_drift_alert_js = (
    uvar_drift_js.loc[:, (slice(None), slice(None), ["start_date", "alert"])]
    .droplevel(level=(1, 2), axis=1)
    .set_index("chunk")
    .rename_axis(mapper="reference_date", axis=0)
)
count_uvar_drift_alert_js = uvar_drift_alert_js.sum(axis=1)
count_uvar_drift_alert_js.name = "count_uvar_drift_alerts"
count_uvar_drift_alert_js

## Multivariate Drift Detection

In [ ]:
reconstruction_drift_calc = nml.DataReconstructionDriftCalculator(
    column_names=features,
    timestamp_column_name=timestamp,
    chunk_period="M",
)

reconstruction_drift = reconstruction_drift_calc.fit(reference).calculate(analysis)
reconstruction_drift.plot().show()

In [ ]:
mvar_reconst_error = reconstruction_drift.filter(
    metrics=["reconstruction_error"]
).to_df()
mvar_reconst_error_alert = (
    mvar_reconst_error.loc[:, (slice(None), ["key", "alert"])]
    .droplevel(level=0, axis=1)
    .set_index("key")
    .rename_axis(mapper="reference_date", axis=0)
)
mvar_reconst_error_alert.index = pd.to_datetime(mvar_reconst_error_alert.index)
mvar_reconst_error_alert.columns = ["mvar_reconst_error_alert"]
mvar_reconst_error_alert

## Monitoring of Prediction Distributions

In [ ]:
reference[col_pred_proba].hist(bins=10, alpha=0.5, label="reference")
analysis[col_pred_proba].hist(bins=10, alpha=0.5, label="analysis")
plt.legend()
plt.show()

## Confidence-Based Performance Estimation (CBPE)

In [ ]:
cbpe_estimator = nml.CBPE(
    problem_type=problem_type,
    y_pred_proba=col_pred_proba,
    y_pred=col_pred,
    y_true=target,
    timestamp_column_name=timestamp,
    metrics=["roc_auc", "precision", "recall"],
    chunk_period="M",
)

cbpe = cbpe_estimator.fit(reference).estimate(analysis)

cbpe.plot().show()

In [ ]:
cbpe_df = cbpe.to_df()
cbpe_est_auc = pd.DataFrame(
    {
        "reference_date": cbpe_df[("chunk", "start_date")],
        "conf_based_perf_est_auc": cbpe_df[("roc_auc", "value")],
    }
).set_index("reference_date")
cbpe_est_auc

## Business KPI Monitoring

In [ ]:
prediction_vs_ground_truth = (
    pd.concat(
        [reference, analysis],
        # keys=["reference", "analysis"],
        # names=["period", "ind"]
    )
    .groupby(
        [
            # "period",
            pd.Grouper(key="reference_date", freq="MS")
        ]
    )[[target, col_pred_proba, col_pred]]
    .mean()
    .sort_index(ascending=(False, True))
)
prediction_vs_ground_truth

## Monitoring Summary

In [ ]:
monitoring_summary = pd.concat(
    [
        prediction_vs_ground_truth,
        cbpe_est_auc,
        count_uvar_drift_alert_js,
        mvar_reconst_error_alert,
    ],
    axis=1,
)
monitoring_summary

## API event monitoring

The same report API can consume serving JSONL logs. It reports request and error volume, latency, and prediction distributions even when feature logging is disabled. Events without feature payloads are excluded from feature-drift calculations.

In [ ]:
from churn_mlops.monitoring import run_monitoring

api_monitor_report = run_monitoring(
    reference_csv=settings.output_dir / "predictions_training.csv",
    prediction_log=settings.prediction_log_path,
    error_log=settings.error_log_path,
    output_dir=settings.output_dir / "monitoring_api",
)
api_monitor_report.summary